<a href="https://colab.research.google.com/github/Unsa15120/ExData_Plotting1/blob/master/NOC_Delay_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Importing required libraries

In [ ]:
#!pip install xgboost -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score

print("Libraries imported successfully")

Libraries imported successfully


##Uploading NOC_Data.xlsx into Colab

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving NOC Data.xlsx to NOC Data.xlsx


##Loading Data into a dataframe

In [ ]:
df = pd.read_excel("NOC Data.xlsx")

print(df.shape)
df.head()

(4686, 6)


,NOC Number,Received \nDate,Outgoing Date,Interface,NOC Type,interface stage
0,1.0,2020-01-02,2020-01-02 00:00:00,Conflict,Design NOC,operational
1,2.0,2020-01-05,2020-01-05 00:00:00,Conflict,Design NOC,under construction
2,3.0,2020-01-05,2020-01-05 00:00:00,Conflict,Design NOC,under construction
3,4.0,2020-01-05,2020-05-19 00:00:00,Conflict,Design NOC,under construction
4,5.0,2020-01-05,2020-04-02 00:00:00,Conflict,Design NOC,under construction


##Checking Column Names & Data Types

In [ ]:
print("COLUMN NAMES:")
print(df.columns.tolist())
print()
print("DATA TYPES:")
print(df.dtypes)

COLUMN NAMES:
['NOC Number', 'Received \nDate', 'Outgoing Date', 'Interface', 'NOC Type', 'interface stage']

DATA TYPES:
NOC Number                float64
Received \nDate    datetime64[ns]
Outgoing Date              object
Interface                  object
NOC Type                   object
interface stage            object
dtype: object


##Checking Missing Values Per Column

In [ ]:
print("MISSING VALUES PER COLUMN:")
print(df.isnull().sum())

MISSING VALUES PER COLUMN:
NOC Number         3951
Received \nDate    1986
Outgoing Date       207
Interface             1
NOC Type           1987
interface stage    1987
dtype: int64


##Checking Inconsistent Text Values in Categorical Columns

In [ ]:
print("INTERFACE column unique values:")
print(df['Interface'].value_counts(dropna=False))
print()
print("NOC TYPE column unique values:")
print(df['NOC Type'].value_counts(dropna=False))
print()
print("INTERFACE STAGE column unique values:")
print(df['interface stage'].value_counts(dropna=False))

INTERFACE column unique values:
Interface
Conflict        2296
no Conflict     1975
No Conflict      404
no conflict        9
NaN                1
 Conflict          1
Name: count, dtype: int64

NOC TYPE column unique values:
NOC Type
NaN                 1987
Information NOC      967
Design NOC           766
Construction NOC     722
Trial Trench NOC     242
Route Approval         2
Name: count, dtype: int64

INTERFACE STAGE column unique values:
interface stage
NaN                                 1987
under construction                  1680
Under Design                         463
under design                         389
operational                          165
under construction /under design       2
Name: count, dtype: int64


##Checking Date Column Format Issues

In [ ]:
print("Received Date sample values and type:")
print(df['Received \nDate'].head(10))
print(df['Received \nDate'].dtype)
print()
print("Outgoing Date sample values and type:")
print(df['Outgoing Date'].head(10))
print(df['Outgoing Date'].dtype)

Received Date sample values and type:
0   2020-01-02
1   2020-01-05
2   2020-01-05
3   2020-01-05
4   2020-01-05
5   2020-01-06
6   2020-01-06
7   2020-01-06
8   2020-01-08
9   2020-01-08
Name: Received \nDate, dtype: datetime64[ns]
datetime64[ns]

Outgoing Date sample values and type:
0    2020-01-02 00:00:00
1    2020-01-05 00:00:00
2    2020-01-05 00:00:00
3    2020-05-19 00:00:00
4    2020-04-02 00:00:00
5    2020-01-06 00:00:00
6    2020-01-14 00:00:00
7    2020-02-19 00:00:00
8    2020-02-05 00:00:00
9    2020-01-22 00:00:00
Name: Outgoing Date, dtype: object
object


##Checking Rows with Missing Critical Fields (NOC Number, Received Date, NOC Type)

In [ ]:
missing_critical = df[df['NOC Number'].isnull() | df['Received \nDate'].isnull() | df['NOC Type'].isnull()]
print("Rows with missing critical fields:", missing_critical.shape[0])
missing_critical.tail(10)

Rows with missing critical fields: 3951


,NOC Number,Received \nDate,Outgoing Date,Interface,NOC Type,interface stage
4676,NaN,NaT,2026-06-20 00:00:00,no Conflict,NaN,NaN
4677,NaN,NaT,2026-06-20 00:00:00,no Conflict,NaN,NaN
4678,NaN,NaT,2026-06-21 00:00:00,no Conflict,NaN,NaN
4679,NaN,NaT,2026-06-21 00:00:00,no Conflict,NaN,NaN
4680,NaN,NaT,2026-06-21 00:00:00,no Conflict,NaN,NaN
4681,NaN,NaT,2026-06-21 00:00:00,no Conflict,NaN,NaN
4682,NaN,NaT,2026-06-21 00:00:00,no Conflict,NaN,NaN
4683,NaN,NaT,2026-06-22 00:00:00,no Conflict,NaN,NaN
4684,NaN,NaT,2026-06-22 00:00:00,no Conflict,NaN,NaN
4685,NaN,NaT,2026-06-22 00:00:00,no Conflict,NaN,NaN


##Checking Negative or Impossible Date Differences

In [ ]:
df_temp = df.copy()
df_temp['Received \nDate'] = pd.to_datetime(df_temp['Received \nDate'], errors='coerce')
df_temp['Outgoing Date'] = pd.to_datetime(df_temp['Outgoing Date'], errors='coerce')
df_temp['check_diff'] = (df_temp['Outgoing Date'] - df_temp['Received \nDate']).dt.days

print("Rows with negative processing days (Outgoing before Received):")
print(df_temp[df_temp['check_diff'] < 0].shape[0])
print()
print("Rows where date conversion failed (NaT):")
print(df_temp['check_diff'].isnull().sum())

Rows with negative processing days (Outgoing before Received):
44

Rows where date conversion failed (NaT):
2193


#Data Cleaning

##Clean Categorical Column Interface

In [ ]:
df['Interface'] = df['Interface'].astype(str).str.strip().str.lower()
df['Interface'] = df['Interface'].replace({
    'conflict': 'Conflict',
    'no conflict': 'No Conflict',
    'nan': np.nan
})

print("Interface values after cleaning:")
print(df['Interface'].value_counts(dropna=False))

Interface values after cleaning:
Interface
No Conflict    2388
Conflict       2297
NaN               1
Name: count, dtype: int64


##Clean Categorical Column interface stage

In [ ]:
df['interface stage'] = df['interface stage'].astype(str).str.strip().str.lower()
df['interface stage'] = df['interface stage'].replace({
    'under construction': 'Under Construction',
    'under design': 'Under Design',
    'operational': 'Operational',
    'under construction /under design': 'Under Construction',
    'nan': np.nan
})

print("Interface stage values after cleaning:")
print(df['interface stage'].value_counts(dropna=False))

Interface stage values after cleaning:
interface stage
NaN                   1987
Under Construction    1682
Under Design           852
Operational            165
Name: count, dtype: int64


##Clean Categorical Column NOC Type

In [ ]:
df['NOC Type'] = df['NOC Type'].astype(str).str.strip().str.lower()
df['NOC Type'] = df['NOC Type'].replace({'nan': np.nan})

print("NOC Type values after cleaning:")
print(df['NOC Type'].value_counts(dropna=False))

NOC Type values after cleaning:
NOC Type
NaN                 1987
information noc      967
design noc           766
construction noc     722
trial trench noc     242
route approval         2
Name: count, dtype: int64


##Fill Missing Categorical Values Using Mode

In [ ]:
interface_mode = df['Interface'].mode()[0]
noc_type_mode = df['NOC Type'].mode()[0]
interface_stage_mode = df['interface stage'].mode()[0]

df['Interface'] = df['Interface'].fillna(interface_mode)
df['NOC Type'] = df['NOC Type'].fillna(noc_type_mode)
df['interface stage'] = df['interface stage'].fillna(interface_stage_mode)

print("Filled Interface with mode:", interface_mode)
print("Filled NOC Type with mode:", noc_type_mode)
print("Filled interface stage with mode:", interface_stage_mode)
print()
print("Missing values remaining in categorical columns:")
print(df[['Interface', 'NOC Type', 'interface stage']].isnull().sum())

Filled Interface with mode: No Conflict
Filled NOC Type with mode: information noc
Filled interface stage with mode: Under Construction

Missing values remaining in categorical columns:
Interface          0
NOC Type           0
interface stage    0
dtype: int64


##Convert Date Columns to Datetime

In [ ]:
df['Received \nDate'] = pd.to_datetime(df['Received \nDate'], errors='coerce')
df['Outgoing Date'] = pd.to_datetime(df['Outgoing Date'], errors='coerce')

print("Date columns converted. Data types now:")
print(df[['Received \nDate', 'Outgoing Date']].dtypes)
print()
print("Missing values in date columns:")
print(df[['Received \nDate', 'Outgoing Date']].isnull().sum())

Date columns converted. Data types now:
Received \nDate    datetime64[ns]
Outgoing Date      datetime64[ns]
dtype: object

Missing values in date columns:
Received \nDate    1986
Outgoing Date       207
dtype: int64


##Drop Rows Missing Outgoing Date

In [ ]:
before_rows = df.shape[0]

df = df.dropna(subset=['Outgoing Date'])

after_rows = df.shape[0]

print(f"Rows before removing missing Outgoing Date: {before_rows}")
print(f"Rows after removing missing Outgoing Date: {after_rows}")
print(f"Rows removed: {before_rows - after_rows}")

Rows before removing missing Outgoing Date: 4686
Rows after removing missing Outgoing Date: 4479
Rows removed: 207


##Fill Missing Received Date Using Median Processing Time per NOC Type

In [ ]:
df['Processing_Days'] = (df['Outgoing Date'] - df['Received \nDate']).dt.days


median_days_by_type = df.groupby('NOC Type')['Processing_Days'].median()
print("Median processing days per NOC Type (used for filling):")
print(median_days_by_type)
print()


missing_received = df['Received \nDate'].isnull()

for noc_type in df['NOC Type'].unique():
    mask = missing_received & (df['NOC Type'] == noc_type)
    median_days = median_days_by_type.get(noc_type, df['Processing_Days'].median())
    df.loc[mask, 'Received \nDate'] = df.loc[mask, 'Outgoing Date'] - pd.to_timedelta(median_days, unit='D')

print("Missing values remaining in Received Date:", df['Received \nDate'].isnull().sum())

Median processing days per NOC Type (used for filling):
NOC Type
construction noc    27.0
design noc          21.0
information noc     25.0
route approval      54.0
trial trench noc    21.0
Name: Processing_Days, dtype: float64

Missing values remaining in Received Date: 0


In [ ]:
before_rows = df.shape[0]

df['Processing_Days'] = (df['Outgoing Date'] - df['Received \nDate']).dt.days
df = df[df['Processing_Days'] >= 0]

after_rows = df.shape[0]

print(f"Rows before removing negative day entries: {before_rows}")
print(f"Rows after removing negative day entries: {after_rows}")
print(f"Rows removed: {before_rows - after_rows}")

Rows before removing negative day entries: 4479
Rows after removing negative day entries: 4435
Rows removed: 44


##Final Check

In [ ]:
print("Final dataset shape:", df.shape)
print()
print("Missing values remaining:")
print(df.isnull().sum())
print()
df.head(10)

Final dataset shape: (4435, 7)

Missing values remaining:
NOC Number         3704
Received \nDate       0
Outgoing Date         0
Interface             0
NOC Type              0
interface stage       0
Processing_Days       0
dtype: int64



,NOC Number,Received \nDate,Outgoing Date,Interface,NOC Type,interface stage,Processing_Days
0,1.0,2020-01-02,2020-01-02,Conflict,design noc,Operational,0
1,2.0,2020-01-05,2020-01-05,Conflict,design noc,Under Construction,0
2,3.0,2020-01-05,2020-01-05,Conflict,design noc,Under Construction,0
3,4.0,2020-01-05,2020-05-19,Conflict,design noc,Under Construction,135
4,5.0,2020-01-05,2020-04-02,Conflict,design noc,Under Construction,88
5,6.0,2020-01-06,2020-01-06,Conflict,design noc,Under Construction,0
6,7.0,2020-01-06,2020-01-14,Conflict,design noc,Under Design,8
7,8.0,2020-01-06,2020-02-19,Conflict,design noc,Under Construction,44
8,9.0,2020-01-08,2020-02-05,Conflict,design noc,Under Construction,28
9,10.0,2020-01-08,2020-01-22,Conflict,design noc,Under Construction,14


In [ ]:
print("Number of duplicate rows (excluding NOC Number):")
print(df.duplicated(subset=df.columns.drop('NOC Number')).sum())

Number of duplicate rows (excluding NOC Number):
2073


In [ ]:
df = df.drop(columns=['NOC Number'])

before_rows = df.shape[0]

df = df.drop_duplicates()

after_rows = df.shape[0]

print(f"Rows before removing duplicates: {before_rows}")
print(f"Rows after removing duplicates: {after_rows}")
print(f"Duplicate rows removed: {before_rows - after_rows}")

Rows before removing duplicates: 4435
Rows after removing duplicates: 2362
Duplicate rows removed: 2073


In [ ]:
target_rows = 3200
np.random.seed(42)


max_received_date = pd.Timestamp("2026-05-28")
max_outgoing_date = pd.Timestamp("2026-06-22")

df_augmented = df.copy()

while len(df_augmented) < target_rows:


    row = df.sample(1).copy()


    received_shift = np.random.randint(-180, 181)
    row["Received \nDate"] = (
        row["Received \nDate"] +
        pd.to_timedelta(received_shift, unit="D")
    )


    if row["Received \nDate"].iloc[0] > max_received_date:
        row["Received \nDate"] = max_received_date


    processing_shift = np.random.randint(-10, 11)

    new_processing = max(
        0,
        int(row["Processing_Days"].iloc[0]) + processing_shift
    )

    row["Processing_Days"] = new_processing


    row["Outgoing Date"] = (
        row["Received \nDate"] +
        pd.to_timedelta(new_processing, unit="D")
    )


    if row["Outgoing Date"].iloc[0] > max_outgoing_date:
        row["Outgoing Date"] = max_outgoing_date
        row["Processing_Days"] = (
            row["Outgoing Date"].iloc[0] -
            row["Received \nDate"].iloc[0]
        ).days


    if np.random.rand() < 0.30:
        row["Interface"] = np.random.choice(df["Interface"].unique())

    if np.random.rand() < 0.30:
        row["NOC Type"] = np.random.choice(df["NOC Type"].unique())

    if np.random.rand() < 0.30:
        row["interface stage"] = np.random.choice(df["interface stage"].unique())


    temp = pd.concat([df_augmented, row], ignore_index=True)

    if len(temp.drop_duplicates()) > len(df_augmented):
        df_augmented = temp.drop_duplicates(ignore_index=True)

print("Original dataset shape :", df.shape)
print("Final dataset shape    :", df_augmented.shape)
print("Received Date max :", df_augmented["Received \nDate"].max())
print("Outgoing Date max :", df_augmented["Outgoing Date"].max())


df_augmented.to_excel("NOC_Data_3200.xlsx", index=False)

print("Dataset saved successfully as NOC_Data_3200.xlsx")

Original dataset shape : (2362, 6)
Final dataset shape    : (3200, 6)
Received Date max : 2026-05-28 00:00:00
Outgoing Date max : 2026-06-22 00:00:00
Dataset saved successfully as NOC_Data_3200.xlsx


In [ ]:
duplicate_rows = df_augmented.duplicated().sum()

print("Number of duplicate rows:", duplicate_rows)

Number of duplicate rows: 0


In [ ]:
#df.to_excel('noc_data.xlsx', index=False)

#print("File saved successfully as 'noc_data.xlsx'")
#print("Shape of exported file:", df.shape)